**Case targets**
- `bad_flg_90plus_24m`: 90+ DPD on this account within 24 months.
- `consumer_bad_flg_90plus_24m`: 90+ DPD on any consumer account within 24 months.

The PDF requires identifying early-warning risk features and creating actionable
underwriting segments. This analysis trains **separate XGBoost models**, evaluates
performance on a held-out test set, and plots global/local SHAP explanations.

**Assumption:** Input is already cleaned. No sentinel-value replacement, cohort
removal, or cleaning rules are repeated here. Models still encode categorical
predictors and allow numeric NaNs, which XGBoost handles natively.

**Important caveat:** Balance/line measurements at 12/24 months are excluded
from the predictive models by default because their timing relative to the
24-month target is unknown. SHAP explains predictions, not causality.

Install packages if needed (run in a terminal):
`python -m pip install pandas numpy scikit-learn xgboost shap matplotlib joblib`

In [1]:
import shap
import xgboost
import sklearn

print("SHAP:", shap.__version__)
print("XGBoost:", xgboost.__version__)
print("Scikit-learn:", sklearn.__version__)

/Users/benjamin/miniconda3/envs/bhdac.1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP: 0.51.0
XGBoost: 3.2.0
Scikit-learn: 1.8.0


In [4]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.calibration import CalibrationDisplay
from xgboost import XGBClassifier

DATA_PATH = Path("clean.csv")
OUTPUT_DIR = Path("xgboost_shap_clean_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
SHAP_SAMPLE_SIZE = 600  # small sample makes SHAP fast, even for 300,000 accounts

TARGETS = ["bad_flg_90plus_24m", "consumer_bad_flg_90plus_24m"]

# Only change this to True after proving these values were known at the decision
# date BEFORE the 24-month default outcome. Otherwise including them may leak.
ALLOW_TRAJECTORY_PREDICTORS = False

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Set DATA_PATH to the location of your already-clean CSV."
    )

## 1. Load your already-clean dataset
No data cleaning or sentinel-value replacement is performed here. We only
verify that the two binary targets exist and have the expected values.
If `test.ipynb` wrote `BHDAC-Fall-26-Case-Comp/clean.csv`, the default path works.

In [5]:
df = pd.read_csv(DATA_PATH, low_memory=False)
missing_targets = [t for t in TARGETS if t not in df.columns]
if missing_targets:
    raise KeyError(f"These outcome columns are missing from the clean CSV: {missing_targets}")

for target in TARGETS:
    actual_values = set(df[target].dropna().unique())
    if not actual_values.issubset({0, 1}):
        raise ValueError(f"{target} must be coded 0/1; observed: {actual_values}")

print(f"Clean accounts loaded: {len(df):,}")
print("Observed outcome rates in clean dataset:")
print(df[TARGETS].mean().rename("Observed rate").to_string())

Clean accounts loaded: 298,963
Observed outcome rates in clean dataset:
bad_flg_90plus_24m             0.062593
consumer_bad_flg_90plus_24m    0.090660


## 2. Engineer credit-line and balance trends (descriptive by default)
These are requested by the case, but cannot safely be used to predict 24-month
defaults until their timestamps and the intended prediction date are documented.

In [6]:
TRAJECTORY_SOURCE_COLS = ["line_12m", "line_24m", "bal_12m", "bal_24m"]
TRAJECTORY_FEATURE_COLS = [
    "line_change_12_to_24m", "balance_change_12_to_24m",
    "utilization_12m", "utilization_24m", "utilization_change_12_to_24m",
]

for col in TRAJECTORY_SOURCE_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if set(TRAJECTORY_SOURCE_COLS).issubset(df.columns):
    df["line_change_12_to_24m"] = df["line_24m"] - df["line_12m"]
    df["balance_change_12_to_24m"] = df["bal_24m"] - df["bal_12m"]
    df["utilization_12m"] = df["bal_12m"] / df["line_12m"].replace(0, np.nan)
    df["utilization_24m"] = df["bal_24m"] / df["line_24m"].replace(0, np.nan)
    df["utilization_change_12_to_24m"] = df["utilization_24m"] - df["utilization_12m"]
    descriptive_cols = [c for c in TRAJECTORY_FEATURE_COLS if c in df.columns]
    print("\nDescriptive trajectory summaries (not automatically used for prediction):")
    print(df[descriptive_cols].describe().round(3).to_string())
    df[descriptive_cols + TARGETS].to_csv(
        OUTPUT_DIR / "descriptive_trajectory_features.csv", index=False
    )
else:
    print("12m/24m balance/line columns not all present; skipped trajectories.")


Descriptive trajectory summaries (not automatically used for prediction):
       line_change_12_to_24m  balance_change_12_to_24m  utilization_12m  utilization_24m  utilization_change_12_to_24m
count             298963.000                298963.000       298935.000       298930.000                    298930.000
mean                 244.259                   -76.965            0.342            0.312                        -0.031
std                 1882.337                  2335.276            0.416            0.430                         0.335
min               -76000.000                -77044.000            0.000            0.000                       -12.130
25%                    0.000                  -153.000            0.000            0.000                        -0.065
50%                    0.000                     0.000            0.108            0.031                         0.000
75%                    0.000                   174.000            0.719            0.665    

## 3. Select safe candidate predictors and make train/validation/test splits
Both targets are *always* removed from features. Numeric missing values remain NaN:
XGBoost handles them directly. Categorical one-hot encoding is fitted only on
training data. A shared test set allows comparing the two default definitions.

The case PDF does not establish as-of dates or unique borrower IDs. If those
fields become available, use a time-based test or group-by-borrower holdout.

In [7]:
model_df = df.dropna(subset=TARGETS).copy()
print(f"\nAccounts with both outcomes available: {len(model_df):,}")

excluded_features = set(TARGETS)
if not ALLOW_TRAJECTORY_PREDICTORS:
    excluded_features.update(TRAJECTORY_SOURCE_COLS)
    excluded_features.update(TRAJECTORY_FEATURE_COLS)

# Remove fields that may be identifiers or explicitly non-predictive.
excluded_features.update(["Acct14", "Acct8", "PubRec1"])

X = model_df.drop(columns=list(excluded_features), errors="ignore").copy()
y = model_df[TARGETS].astype("int8").copy()

# Drop constant and entirely absent features; nothing useful to learn from them.
constant_features = [c for c in X if X[c].nunique(dropna=False) <= 1]
X = X.drop(columns=constant_features)

categorical_cols = list(X.select_dtypes(include=["object", "category", "string"]).columns)
numeric_cols = [c for c in X if c not in categorical_cols]

for col in numeric_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
X[numeric_cols] = X[numeric_cols].replace([np.inf, -np.inf], np.nan)
for col in categorical_cols:
    X[col] = X[col].astype("object").fillna("__MISSING__").astype(str)

if X.shape[1] == 0:
    raise ValueError("No usable predictors remain after feature selection.")

# Use joint target categories for stratification where possible.
joint_strata = (2 * y[TARGETS[0]] + y[TARGETS[1]]).astype(int)
strata = joint_strata if joint_strata.value_counts().min() >= 5 else y[TARGETS[0]]
positions = np.arange(len(X))
trainval_pos, test_pos = train_test_split(
    positions, test_size=0.20, random_state=RANDOM_STATE, stratify=strata
)
train_pos, val_pos = train_test_split(
    trainval_pos, test_size=0.25, random_state=RANDOM_STATE,
    stratify=strata.iloc[trainval_pos]
)

try:
    category_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # older scikit-learn
    category_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

transformers = []
if numeric_cols:
    transformers.append(("numeric", "passthrough", numeric_cols))
if categorical_cols:
    transformers.append(("category", category_encoder, categorical_cols))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    sparse_threshold=0,
    verbose_feature_names_out=False,
)

preprocessor.fit(X.iloc[train_pos])
feature_names = preprocessor.get_feature_names_out().tolist()


def make_model_matrix(pos):
    matrix = preprocessor.transform(X.iloc[pos]).astype(np.float32)
    return pd.DataFrame(matrix, columns=feature_names, index=X.iloc[pos].index)


X_train = make_model_matrix(train_pos)
X_val = make_model_matrix(val_pos)
X_test = make_model_matrix(test_pos)
print(f"\nTrain: {len(train_pos):,} | Validation: {len(val_pos):,} | Test: {len(test_pos):,}")
print(f"Model features after encoding: {len(feature_names)}")
print("Train default rates:", y.iloc[train_pos].mean().round(4).to_dict())


Accounts with both outcomes available: 298,963

Train: 179,377 | Validation: 59,793 | Test: 59,793
Model features after encoding: 59
Train default rates: {'bad_flg_90plus_24m': 0.0626, 'consumer_bad_flg_90plus_24m': 0.0907}


## 4. Fit independent XGBoost models for account and consumer default
We tune the stopping iteration on a separate validation split. The untouched
test set estimates out-of-sample discrimination, precision-recall performance,
and probability accuracy. Average precision should be read alongside event rate.

In [8]:
def top_decile_stats(y_true, probability):
    observed = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)
    n_top = max(1, int(np.ceil(len(probability) * 0.10)))
    indices = np.argsort(-probability)[:n_top]
    overall_rate = observed.mean()
    top_rate = observed[indices].mean()
    return {
        "top_decile_observed_rate": top_rate,
        "top_decile_lift": top_rate / overall_rate if overall_rate > 0 else np.nan,
        "share_of_defaults_in_top_decile": (
            observed[indices].sum() / observed.sum() if observed.sum() > 0 else np.nan
        ),
    }


models = {}
test_probabilities = {}
metric_rows = []
heldout = model_df.iloc[test_pos][TARGETS].copy()

for target in TARGETS:
    print(f"\n===== Training {target} =====")
    y_train = y.iloc[train_pos][target]
    y_val = y.iloc[val_pos][target]
    y_test = y.iloc[test_pos][target]

    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=700,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=15,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=8.0,
        reg_alpha=0.5,
        tree_method="hist",
        eval_metric="aucpr",
        early_stopping_rounds=40,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    probability = model.predict_proba(X_test)[:, 1]

    metrics = {
        "target": target,
        "n_train": len(y_train),
        "n_validation": len(y_val),
        "n_test": len(y_test),
        "observed_test_default_rate": y_test.mean(),
        "roc_auc": roc_auc_score(y_test, probability),
        "average_precision_pr_auc": average_precision_score(y_test, probability),
        "brier_score_lower_is_better": brier_score_loss(y_test, probability),
        "best_boosting_iteration": model.best_iteration,
    }
    metrics.update(top_decile_stats(y_test, probability))
    metric_rows.append(metrics)
    models[target] = model
    test_probabilities[target] = probability
    heldout[f"pred_{target}"] = probability

    print(pd.Series(metrics).to_string())
    joblib.dump(
        {"model": model, "preprocessor": preprocessor,
         "raw_feature_columns": X.columns.tolist(), "encoded_feature_names": feature_names,
         "categorical_columns": categorical_cols,
         "trajectory_predictors_allowed": ALLOW_TRAJECTORY_PREDICTORS},
        OUTPUT_DIR / f"xgboost_{target}.joblib",
    )

metrics_table = pd.DataFrame(metric_rows)
metrics_table.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
heldout.to_csv(OUTPUT_DIR / "holdout_predictions.csv", index=True)
print("\nModel comparison:")
print(metrics_table.round(4).to_string(index=False))


===== Training bad_flg_90plus_24m =====
target                             bad_flg_90plus_24m
n_train                                        179377
n_validation                                    59793
n_test                                          59793
observed_test_default_rate                   0.062599
roc_auc                                      0.859081
average_precision_pr_auc                     0.328724
brier_score_lower_is_better                   0.04915
best_boosting_iteration                           437
top_decile_observed_rate                     0.308194
top_decile_lift                              4.923281
share_of_defaults_in_top_decile              0.492386

===== Training consumer_bad_flg_90plus_24m =====
target                             consumer_bad_flg_90plus_24m
n_train                                                 179377
n_validation                                             59793
n_test                                                   59793
observed_

## 5. Evaluate discrimination and calibration on the untouched test data
ROC AUC measures ranking. Average precision is especially informative for a
relatively rare event. Brier score and the calibration plot assess whether
the predicted probabilities correspond to actual default frequencies.

In [ ]:
for target in TARGETS:
    y_test = y.iloc[test_pos][target]
    probability = test_probabilities[target]
    tag = "account" if target == TARGETS[0] else "consumer"

    RocCurveDisplay.from_predictions(y_test, probability, name=tag)
    plt.title(f"ROC curve — {tag} 90+ DPD")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"roc_{tag}.png", dpi=160, bbox_inches="tight")
    plt.show()

    PrecisionRecallDisplay.from_predictions(y_test, probability, name=tag)
    plt.axhline(y_test.mean(), ls="--", lw=1, color="gray", label="Event rate")
    plt.title(f"Precision–recall curve — {tag} 90+ DPD")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"precision_recall_{tag}.png", dpi=160, bbox_inches="tight")
    plt.show()

    CalibrationDisplay.from_predictions(
        y_test, probability, n_bins=10, strategy="quantile", name=tag
    )
    plt.title(f"Probability calibration — {tag} 90+ DPD")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"calibration_{tag}.png", dpi=160, bbox_inches="tight")
    plt.show()

## 6. Interpret each model with SHAP (global and individual explanations)
- **SHAP bar:** overall average absolute contribution; ranks features.
- **SHAP beeswarm:** impact and direction across customers. Positive values
  indicate higher model-predicted default risk *on the log-odds scale*.
- **SHAP waterfall:** explains one high-scoring test account's prediction.

These are associations inside the model, not causal effects. In particular,
credit-line policy cannot be changed solely on the basis of a SHAP ranking.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample_pos = np.sort(
    rng.choice(len(X_test), size=min(SHAP_SAMPLE_SIZE, len(X_test)), replace=False)
)
X_shap = X_test.iloc[sample_pos].copy()

for target in TARGETS:
    tag = "account" if target == TARGETS[0] else "consumer"
    model = models[target]
    print(f"\n===== SHAP explanations: {tag} default =====")

    explainer = shap.TreeExplainer(model)
    explanations = explainer(X_shap)
    if np.asarray(explanations.values).ndim == 3:
        # Some binary classifier explainers include a separate output axis.
        explanations = explanations[:, :, 1]

    importance = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap_log_odds": np.abs(explanations.values).mean(axis=0),
    }).sort_values("mean_abs_shap_log_odds", ascending=False)
    importance.to_csv(OUTPUT_DIR / f"shap_importance_{tag}.csv", index=False)
    print("Top drivers of model prediction (mean absolute SHAP):")
    print(importance.head(15).round(4).to_string(index=False))

    shap.plots.bar(explanations, max_display=20, show=False)
    plt.title(f"SHAP importance — {tag} 90+ DPD")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"shap_bar_{tag}.png", dpi=160, bbox_inches="tight")
    plt.show()

    shap.plots.beeswarm(explanations, max_display=20, show=False)
    plt.title(f"SHAP impact and direction — {tag} 90+ DPD")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"shap_beeswarm_{tag}.png", dpi=160, bbox_inches="tight")
    plt.show()

    # Select the highest predicted-risk observation *within the SHAP sample*.
    sample_pd = test_probabilities[target][sample_pos]
    example_pos = int(np.argmax(sample_pd))
    print(f"Example held-out predicted {tag} default probability: "
          f"{sample_pd[example_pos]:.2%}")
    shap.plots.waterfall(explanations[example_pos], max_display=16, show=False)
    plt.title(f"Single-account SHAP explanation — {tag} 90+ DPD")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"shap_waterfall_{tag}.png", dpi=160, bbox_inches="tight")
    plt.show()

## 7. Illustrative four-tier underwriting analysis (account model)
These probability ranges are example thresholds, *not* defined by the PDF.
Validate thresholds against observed defaults and actual revenues, loss-given-
default, costs, and credit-exposure constraints before making policy decisions.

In [ ]:
heldout["account_risk_tier"] = pd.cut(
    heldout[f"pred_{TARGETS[0]}"],
    bins=[-np.inf, 0.03, 0.08, 0.15, np.inf],
    labels=["Prime", "Near-Prime", "Subprime", "High Risk"],
    right=False,
)

tier_report = (
    heldout.groupby("account_risk_tier", observed=False)
    .agg(
        accounts=(TARGETS[0], "size"),
        avg_predicted_account_pd=(f"pred_{TARGETS[0]}", "mean"),
        observed_account_default_rate=(TARGETS[0], "mean"),
        observed_consumer_default_rate=(TARGETS[1], "mean"),
        avg_predicted_consumer_pd=(f"pred_{TARGETS[1]}", "mean"),
    )
    .reset_index()
)
tier_report["share_of_test_accounts"] = tier_report["accounts"] / len(heldout)
print("\nIllustrative account risk tiers (held-out test set only):")
print(tier_report.round(4).to_string(index=False))
tier_report.to_csv(OUTPUT_DIR / "risk_tier_report.csv", index=False)
heldout.to_csv(OUTPUT_DIR / "holdout_predictions_with_tiers.csv", index=True)

print(f"\nFinished. Results and figures saved in: {OUTPUT_DIR.resolve()}")

## How to write up the findings after running
1. Report sample sizes and observed event rates in the clean data. If your
   earlier cleaning removed no-credit-file accounts, state that exclusion.
2. Compare test ROC AUC, average precision, Brier score, and top-decile lift
   for account and consumer outcomes; never report training metrics as holdout.
3. Insert both SHAP beeswarm plots and discuss leading predictors **without
   claiming causation**. A feature with positive SHAP raises model log-odds.
4. Summarize test-set observed default rates across the four illustrative tiers.
5. State that true **profit optimization** needs revenue, exposure, LGD, and
   funding/operating-cost assumptions that the case PDF does not supply.
6. Before any real underwriting use: verify feature timestamps and cleaning
   assumptions, assess calibration/fairness/stability, and use borrower- or
   time-aware validation when identifiers/dates are available.